# EXAONE-4.5-33B · SSAFY AI 챌린지 · Colab H100

H100 80GB 런타임에서 **런타임 → 모두 실행**을 선택하세요. 드라이브 연결 승인 후 데이터 탐색, BF16 모델 로딩, 이미지 객관식 추론, 제출 파일 저장을 진행합니다.

- 데이터는 `test.csv`, `sample_submission.csv`, `test/`가 함께 있는 폴더입니다. 자동 탐색에 실패하면 코드 상단의 `DATA_DIR_OVERRIDE`만 수정하세요.
- 기존 세션에 다른 라이브러리 버전이 로드되어 있으면 안내에 따라 세션을 다시 시작한 후 모두 실행하세요.
- 모델 선택을 위한 학습 없는 첫 추론 구성입니다. 중간 결과는 10개마다 Drive에 저장하며 재실행 시 이어갑니다. 프롬프트나 설정을 변경하면 `RUN_NAME`도 변경하세요.
- 원본 모델을 Colab 로컬 캐시에 다운로드하므로 충분한 로컬 디스크 공간이 필요합니다. 런타임 삭제 시 캐시도 사라집니다.
- 출력: 데이터 폴더의 `exaone_results/exaone45_33b_bf16_nonreason_v1/submission.csv`
- 프로젝트의 Colab 베이스라인과 CSV 구조에 맞춰 작성했습니다. H100에서 실제 모델 실행은 검증하지 않았습니다.


In [ ]:
# Colab에서 이 셀을 실행하세요. Drive에 데이터가 있으면 자동으로 찾습니다.
import sys
import subprocess
import importlib.metadata
from pathlib import Path

# Google Colab / cuDF와 호환되는 pandas 버전을 유지합니다.
packages_before = {}
for name in ("pandas", "transformers", "accelerate"):
    if name in sys.modules:
        packages_before[name] = getattr(sys.modules[name], "__version__", None)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=5.8.0,<6", "accelerate", "pillow", "tqdm",
    "pandas==2.2.3",
])
for name, old_version in packages_before.items():
    if old_version != importlib.metadata.version(name):
        raise RuntimeError(
            f"{name} 버전이 변경되었습니다. 런타임 → 세션 다시 시작 후 "
            "이 셀을 다시 실행하세요. 설치를 반복해도 괜찮습니다."
        )

from google.colab import drive
drive.mount('/content/drive')

import json
import re
import shutil
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText

# 자동 탐색에 실패할 때만 실제 데이터 폴더를 입력하세요.
# 예: "/content/drive/MyDrive/ai_2_challenge/data"
DATA_DIR_OVERRIDE = ""
MODEL_ID = "LGAI-EXAONE/EXAONE-4.5-33B"
MAX_NEW_TOKENS = 32
RUN_NAME = "exaone45_33b_bf16_nonreason_v1"

my_drive = Path("/content/drive/MyDrive")
candidates = [
    my_drive / "ai_2_challenge/data",
    my_drive / "ai_2_challenge/ssafy-16-2-ai",
    my_drive / "ssafy-16-2-ai",
    my_drive / "data",
    my_drive,
]
if DATA_DIR_OVERRIDE:
    candidates = [Path(DATA_DIR_OVERRIDE)]

def is_dataset_folder(path):
    return (
        (path / "test.csv").is_file()
        and (path / "sample_submission.csv").is_file()
        and (path / "test").is_dir()
    )

DATA_DIR = next((p for p in candidates if is_dataset_folder(p)), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "test.csv, sample_submission.csv, test 폴더가 함께 있는 위치를 "
        "DATA_DIR_OVERRIDE에 입력한 뒤 다시 실행하세요."
    )

OUT_DIR = DATA_DIR / "exaone_results" / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)
test_df = pd.read_csv(DATA_DIR / "test.csv", dtype={"id": str})
submission = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"id": str})

required_columns = {"id", "path", "question", "a", "b", "c", "d"}
assert required_columns.issubset(test_df.columns), "test.csv 열 구성이 다릅니다."
assert test_df["id"].notna().all() and test_df["id"].is_unique
assert submission["id"].notna().all() and submission["id"].is_unique
assert set(submission["id"]) == set(test_df["id"]), "제출 양식과 테스트 ID가 다릅니다."

def image_path(value):
    relative = str(value).replace("\\", "/")
    if relative.startswith("/content/"):
        relative = relative[len("/content/"):]
    path = DATA_DIR / relative
    if not path.is_file():
        raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {path}")
    return path

# 대용량 모델을 받기 전에 전체 이미지 경로를 검사합니다.
for value in test_df["path"]:
    image_path(value)

assert torch.cuda.is_available(), "H100 GPU 런타임을 선택하세요."
gpu = torch.cuda.get_device_properties(0)
print(f"데이터: {DATA_DIR}")
print(f"테스트: {len(test_df)}개 | GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
if gpu.total_memory / 2**30 < 75:
    raise RuntimeError("이 BF16 구성은 H100 80GB 기준입니다. GPU 메모리를 확인하세요.")

checkpoint = OUT_DIR / "test_predictions.csv"
manifest_path = OUT_DIR / "run_config.json"
import hashlib
run_config = {
    "model": MODEL_ID,
    "dtype": "bfloat16",
    "enable_thinking": False,
    "max_new_tokens": MAX_NEW_TOKENS,
    "test_sha256": hashlib.sha256((DATA_DIR / "test.csv").read_bytes()).hexdigest(),
}
if manifest_path.exists():
    if json.loads(manifest_path.read_text()) != run_config:
        raise RuntimeError("저장 결과의 설정이 다릅니다. RUN_NAME을 변경하세요.")
else:
    if checkpoint.exists():
        raise RuntimeError("설정 기록 없는 체크포인트입니다. RUN_NAME을 변경하세요.")
    manifest_path.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

records = (
    pd.read_csv(checkpoint, dtype={"id": str}, keep_default_na=False).to_dict("records")
    if checkpoint.exists() else []
)
done = {r["id"] for r in records}
assert len(done) == len(records), "체크포인트에 중복 ID가 있습니다."
assert done.issubset(set(test_df["id"])), "체크포인트의 ID가 다릅니다."
assert all(r["answer"] in ("a", "b", "c", "d") for r in records)

def save_progress():
    temp = checkpoint.with_suffix(".tmp.csv")
    pd.DataFrame(records, columns=["id", "answer", "raw"]).to_csv(temp, index=False)
    temp.replace(checkpoint)

if len(done) < len(test_df):
    # 같은 세션에서 재실행할 경우 GPU에 모델을 두 번 로딩하지 않습니다.
    if globals().get("_loaded_model_id") != MODEL_ID:
        free_gib = shutil.disk_usage("/content").free / 2**30
        print(f"Colab 로컬 디스크 여유: {free_gib:.1f} GiB")
        print("EXAONE 원본 가중치를 다운로드합니다. 가중치 용량은 약 66GB입니다.")
        processor = AutoProcessor.from_pretrained(MODEL_ID)
        model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID,
            dtype=torch.bfloat16,
            device_map={"": 0},
            attn_implementation="sdpa",
        ).eval()
        _loaded_model_id = MODEL_ID

    @torch.inference_mode()
    def predict(row):
        with Image.open(image_path(row["path"])) as source:
            image = source.convert("RGB")

        prompt = (
            f"{row['question']}\n"
            f"(a) {row['a']}\n(b) {row['b']}\n"
            f"(c) {row['c']}\n(d) {row['d']}\n\n"
            "이미지를 보고 정답을 고르세요. 설명 없이 "
            "a, b, c, d 중 소문자 한 글자만 출력하세요."
        )
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True,
            enable_thinking=False,
        )
        inputs = processor(
            text=[text], images=[image], return_tensors="pt",
        ).to(model.device)

        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        )
        raw = processor.batch_decode(
            output[:, inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True,
        )[0].strip()

        choices = re.findall(r"(?<![a-z])[a-d](?![a-z])", raw.lower())
        if len(set(choices)) != 1:
            raise ValueError(f"{row['id']}: 답변 해석 실패: {raw!r}")
        return choices[0], raw

    print(f"완료 {len(done)}개 / 남은 {len(test_df) - len(done)}개")
    try:
        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="EXAONE 추론"):
            if row["id"] in done:
                continue
            answer, raw = predict(row)
            records.append({"id": row["id"], "answer": answer, "raw": raw})
            done.add(row["id"])
            if len(records) % 10 == 0:
                save_progress()
    finally:
        # 오류나 수동 중단 시에도 그전까지의 결과를 보존합니다.
        save_progress()

predictions = pd.DataFrame(records).set_index("id")["answer"]
submission["answer"] = submission["id"].map(predictions)
assert submission["answer"].isin(["a", "b", "c", "d"]).all(), "누락된 답변이 있습니다."
output_path = OUT_DIR / "submission.csv"
submission.to_csv(output_path, index=False)
print(f"\n완료! 제출 파일: {output_path}")
display(submission.head())
